# Get URLs from Sitemap

In [1]:
import requests
import xml.etree.ElementTree as ET

DBT_SITEMAP_URL = "https://www.getdbt.com/sitemap-0.xml"
DBT_DOC_SITEMAP_URL = "https://docs.getdbt.com/sitemap.xml"

def get_urls_from_sitemap(sitemap_url: str) -> list[str]:
    sitemap_page = requests.get(sitemap_url)
    root = ET.fromstring(sitemap_page.content)
    sitemap_urls = [child[0].text.strip() for child in root]
    return sitemap_urls

In [2]:
dbt_document_urls = get_urls_from_sitemap(DBT_SITEMAP_URL) + get_urls_from_sitemap(DBT_DOC_SITEMAP_URL)

In [3]:
print(len(dbt_document_urls))

2534


In [4]:
dbt_document_urls[:30]

['https://www.getdbt.com/blog',
 'https://www.getdbt.com/case-studies',
 'https://www.getdbt.com/events',
 'https://www.getdbt.com/press',
 'https://www.getdbt.com/resources',
 'https://www.getdbt.com/resources/webinars',
 'https://www.getdbt.com/fr/resources/webinars',
 'https://www.getdbt.com/jp/resources/webinars',
 'https://www.getdbt.com/fr/events',
 'https://www.getdbt.com/jp/events',
 'https://www.getdbt.com',
 'https://www.getdbt.com/2025-dbt-cloud-launch-showcase-recording',
 'https://www.getdbt.com/2025-state-of-analytics-engineering-virtual-event-recording',
 'https://www.getdbt.com/about-us',
 'https://www.getdbt.com/about-us/careers',
 'https://www.getdbt.com/about-us/dei',
 'https://www.getdbt.com/about-us/values',
 'https://www.getdbt.com/boost-ai-reliability-with-dbt-cloud-and-snowflake-cortex-confirmation',
 'https://www.getdbt.com/build-for-scale-agility-and-reliability-with-dbt-cloud-automatedv-and-data-vault/confirmation',
 'https://www.getdbt.com/certifications/ana

# Crawl document

In [75]:
import nest_asyncio
nest_asyncio.apply()

In [76]:
"""
Copyright (c) Django Software Foundation and individual contributors.
https://github.com/django/django/blob/1a744343999c9646912cee76ba0a2fa6ef5e6240/django/utils/text.py#L453-L470
"""
import re
import unicodedata

def slugify(value, allow_unicode=False):
    """
    Convert to ASCII if 'allow_unicode' is False. Convert spaces or repeated
    dashes to single dashes. Remove characters that aren't alphanumerics,
    underscores, or hyphens. Convert to lowercase. Also strip leading and
    trailing whitespace, dashes, and underscores.
    """
    value = str(value)
    if allow_unicode:
        value = unicodedata.normalize("NFKC", value)
    else:
        value = (
            unicodedata.normalize("NFKD", value)
            .encode("ascii", "ignore")
            .decode("ascii")
        )
    value = re.sub(r"[^\w\s-]", "", value.lower())
    return re.sub(r"[-\s]+", "-", value).strip("-_")

In [77]:
def extract_main_content(markdown: str) -> str:
    """
    Remove header and footer for www.getdbt.com
    https://github.com/unclecode/crawl4ai/discussions/508#discussioncomment-11897822
    https://github.com/unclecode/crawl4ai/discussions/508#discussioncomment-11897816
    """
    start_marker = markdown.find('#')
    start_index = 0 if start_marker == -1 else start_marker
    footer_index = markdown.find("Last modified on:")
    if footer_index == -1:
        return markdown[start_index:]
    else:
        return markdown[start_index:footer_index]


def extract_main_content_for_doc(markdown: str) -> str:
    """
    Remove header and footer for doc.getdbt.com
    """
    start_marker = "[Frequently asked questions](https://docs.getdbt.com/docs/faqs)"
    marker_index = markdown.find(start_marker)
    start_index = 0 if marker_index == -1 else marker_index + len(start_marker) + 1
    footer_index = markdown.find("[dbt Cloud Status]")
    if footer_index == -1:
        return markdown[start_index:]
    else:
        return markdown[start_index:footer_index]

In [90]:
import asyncio
import os
from crawl4ai import AsyncWebCrawler, BrowserConfig, CrawlerRunConfig, CacheMode
from crawl4ai.content_filter_strategy import PruningContentFilter
from crawl4ai.markdown_generation_strategy import DefaultMarkdownGenerator

browser_config = BrowserConfig(
    verbose=False
)

# https://docs.crawl4ai.com/core/fit-markdown/
prune_filter = PruningContentFilter(
    threshold=0.48,
    threshold_type="dynamic",  
    min_word_threshold=5      
)

md_generator = DefaultMarkdownGenerator(content_filter=prune_filter)

run_config = CrawlerRunConfig(
    verbose=False,
    word_count_threshold=10,
    # # excluded_tags=["header", "footer"],
    # excluded_selector=["div.footer__items"],
    exclude_external_links=True,
    process_iframes=True,
    remove_overlay_elements=True,
    cache_mode=CacheMode.ENABLED,
    markdown_generator=md_generator
)

async def main(url: str):
    async with AsyncWebCrawler(config=browser_config) as crawler:
        result = await crawler.arun(
            url=url,
            config=run_config
        )

        if result.success:
            raw = result.markdown.raw_markdown
            fit = result.markdown.fit_markdown
            markdown = raw if len(fit) == 0 else fit
            main_content = extract_main_content_for_doc(markdown) if "docs.getdbt.com" in result.url else extract_main_content(markdown)
            fixed_markdown = f"""# Metadata            
- url : {url}
- title : {result.metadata['title']}
- description : {result.metadata['description']}

---

{main_content}"""

            file_path = os.path.join("./crawl4ai_markdown", slugify(result.metadata['title']) + ".md")
            with open(file_path, "w") as f:
                f.write(fixed_markdown)
            
        else:
            print(f"Crawl failed: {result.error_message}")

In [103]:
asyncio.run(
    main(url="https://www.getdbt.com/blog/dbt-canvas-is-ga")
)

In [109]:
%%time

for url in dbt_document_urls:
    asyncio.run(main(url=url))

CPU times: user 1min 13s, sys: 38.2 s, total: 1min 52s
Wall time: 27min 42s


# Document load

In [1]:
from langchain_community.document_loaders import DirectoryLoader, UnstructuredMarkdownLoader

loader = DirectoryLoader(
    "./crawl4ai_markdown", 
    glob="**/*.md", 
    loader_cls=UnstructuredMarkdownLoader,
    show_progress=True,
    use_multithreading=True
)

docs = loader.load()

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▌| 2458/2465 [00:46<00:00, 52.92it/s]


In [2]:
len(docs)

2458

In [4]:
docs[200]

Document(metadata={'source': 'crawl4ai_markdown/one-doc-tagged-with-v1102-dbt-developer-hub.md'}, page_content='Metadata\n\nurl : https://docs.getdbt.com/tags/v-1-1-02\n\ntitle : One doc tagged with "v1.1.02" | dbt Developer Hub\n\ndescription : None\n\nSkip to main content Don\'t miss the 2025 dbt Launch Showcase on May 28 and 29th! Catch live demos and get a first look at the latest features coming to dbt.\n\ndbt Logo\n\nDocs * Product docs * API docs * Best practices * Release notes\n\nGuidesReference Latest (dbt Core v1.10) * dbt Fusion engine (v2.0 Beta) * Latest (dbt Core v1.10) * Compatible (dbt Core v1.9)\n\nResources * Courses * Best practices * Developer blog\n\nCommunity * Join the dbt Community * Become a contributor * Community forum * Events * Spotlight\n\nCreate a free account Search⌘``K\n\nOne doc tagged with "v1.1.02"\n\nView all tags## Changelog 2019 and 2020 2019 and 2020 Changelog for the dbt Cloud application')

In [6]:
from langchain_text_splitters import MarkdownTextSplitter

text_splitter = MarkdownTextSplitter(chunk_size=2000, chunk_overlap=100)

dbt_doc_markdown_texts = text_splitter.split_documents(docs)

In [7]:
len(dbt_doc_markdown_texts)

11928

In [8]:
dbt_doc_markdown_texts[100]

Document(metadata={'source': 'crawl4ai_markdown/constraints-dbt-developer-hub.md'}, page_content='Note that Redshift limits the maximum length of the varchar values to 256 characters by default (or when specified without a length). This means that any string data exceeding 256 characters might get truncated or return a "value too long for character type" error. To allow the maximum length, use varchar(max). For example, data_type: varchar(max). Expected DDL to enforce constraints: target/run/.../constraints_example.sql createtable"database_name"."schema_name"."constraints_example__dbt_tmp"( id integernotnull, customer_name varchar, first_transaction_date date,primarykey(id));insertinto"database_name"."schema_name"."constraints_example__dbt_tmp"(select1as id,\'My Favorite Customer\'as customer_name, cast(\'2019-01-01\'asdate)as first_transaction_date);\n\nSnowflake constraints documentation:\n\nSnowflake data types:\n\nSnowflake supports four types of constraints: unique, not null, prim

# Embedding

In [21]:
from langchain_chroma import Chroma
from langchain_aws import BedrockEmbeddings

config = {
    "region_name": "us-west-2",
    "embedding_model_id": "amazon.titan-embed-text-v2:0",
    "temperature": 0.4
}

embeddings = BedrockEmbeddings(
    model_id=config["embedding_model_id"], region_name=config["region_name"], client=None
)

In [11]:
vector_store = Chroma(
    collection_name="dbt_all_docs_markdown",
    embedding_function=embeddings,
    persist_directory="./chroma_dbt_docs_markdown",
)

Chrome では一度に embedding 可能な上限があるようだ

https://github.com/chroma-core/chroma/issues/1079

In [19]:
from itertools import batched

CHROMA_MAX_BATCH_SIZE = 5461

for batch_texts in batched(dbt_doc_markdown_texts, CHROMA_MAX_BATCH_SIZE):
    print(len(batch_texts))

5461
5461
1006


In [20]:
for batch_texts in batched(dbt_doc_markdown_texts, CHROMA_MAX_BATCH_SIZE):
    vector_store.add_documents(batch_texts)

In [22]:
!tar zcf chroma_dbt_docs_markdown.tar.gz ./chroma_dbt_docs_markdown